# US equities panel: mixing along time and across features instead of recurring

[`06_linear`](06_linear.ipynb), [`07_gbm`](07_gbm.ipynb) and
[`08_tabular_dl`](08_tabular_dl.ipynb) all read the same flat table: one row per stock per
session, one column per feature, and nothing in the representation saying the rows are ordered
in time. A model on that table sees the past only through columns somebody computed in advance -
a 21-session momentum, a rolling volatility. It never sees the sequence itself.

A **sequence model** is handed the sequence. Each training example here is a **window**: the 60
most recent sessions of one stock's features, in order, as a matrix of sessions by features -
about three months. The model reads the window and emits one number, the predicted return.

**A window has to be 60 consecutive sessions of the same stock, and on this panel that binds.**
A stock that lists part-way through a fold, halts, or delists leaves a gap, and a window
spanning a gap would treat the two sides as consecutive sessions and read the jump across it as
one day's move. Windows are therefore built only where the sessions are unbroken, which is why
the number of training examples is far smaller than the number of rows and differs between
folds.

[`10_dl_lstm`](10_dl_lstm.ipynb) walked the window one session at a time. **TSMixer** does not
walk it at all. The window is a matrix of sessions by features, and the model alternates two
operations on it: a small network applied down each feature's column, mixing **across time**,
and a small network applied across each session's row, mixing **across features**. Stacking a
few of those blocks lets information travel along both axes without any recurrence.


<!-- ML4T_ZH_BEGIN c0000-p000 -->
# 美國股票面板資料：沿時間與特徵軸混合資訊，取代遞迴

[`06_linear`](06_linear.ipynb)、[`07_gbm`](07_gbm.ipynb) 與 [`08_tabular_dl`](08_tabular_dl.ipynb) 都讀取相同的平面表格：每檔股票每個交易日一列，每項特徵一欄；資料表示中沒有任何內容指出各列按時間排序。使用這張表格的模型，只透過事先計算好的欄位看到過去，例如 21 個交易日動能或滾動波動率；它從未看到序列本身。

**序列模型**則直接取得序列。此處每筆訓練樣本是一個**窗口**：某檔股票最近 60 個交易日的特徵依時間排序，組成以交易日為列、特徵為欄的矩陣，約涵蓋三個月。模型讀取窗口，輸出一個數字，即預測報酬率。

**窗口必須包含同一檔股票連續 60 個交易日，而這項條件在此面板上確實形成限制。** 股票若在某折中途上市、停牌或下市，就會留下缺口；跨越缺口的窗口會將兩側誤認為連續交易日，把跨越缺口的跳動當成單日變動。因此，只有交易日連續不中斷時才建立窗口，這也是訓練樣本數遠少於資料列數、且各折數量不同的原因。

[`10_dl_lstm`](10_dl_lstm.ipynb) 每次沿窗口前進一個交易日。**TSMixer** 則完全不逐步走訪。窗口是以交易日為列、特徵為欄的矩陣；模型輪流對它執行兩種運算：將小型網路套用於每項特徵的縱欄，**跨時間**混合；再將小型網路套用於每個交易日的橫列，**跨特徵**混合。堆疊數個這樣的區塊，資訊就能沿兩條軸線傳遞，而無須遞迴。
<!-- ML4T_ZH_END c0000-p000 -->

**The reason to try it is the shape of this particular panel.** Recurrence processes a window
sequentially, so its cost grows with the window length and its gradient has to survive the whole
walk; mixing touches the whole window at once.

The other two models do combine features - the LSTM's gates read the whole feature vector at
every step, and NLinear's final layer is a linear combination across features - but each does it
in one place and one way. TSMixer makes feature mixing an explicit, nonlinear operation that
happens once per block and alternates with mixing along time, so a combination of features can
itself be mixed across time and then recombined. On a feature set this dense in the
cross-section, that repetition is what is being tested.

The configuration here stacks two blocks at a hidden dimension of 32, smaller than the recurrent
model's state for the same reason: what bounds the capacity worth fitting is the number of
eligible windows.

**Learning objectives.** By the end of this notebook you will be able to:

- Describe the two mixing operations, say which axis of the window each acts on, and say what
  information can travel where after two blocks.
- Say how TSMixer's feature mixing differs from the way the other two models combine features,
  given that both of them also do.
- Explain why an architecture without recurrence can be preferable on long windows, in terms of
  cost and of what has to survive training.
- Read the epoch schedule out of a declared configuration and say how many scoreable models the
  run publishes for it.


<!-- ML4T_ZH_BEGIN c0000-p001 -->
**嘗試此架構的理由，來自這份面板資料的形態。** 遞迴會依序處理窗口，因此成本隨窗口長度增加，梯度也必須穿過整段序列；混合則一次處理完整窗口。

另外兩種模型也會組合特徵：LSTM 的閘門在每步讀取完整特徵向量，NLinear 的最終層則是跨特徵的線性組合；但各自都只在一個位置、以一種方式完成。TSMixer 將特徵混合明確設為非線性運算，在每個區塊執行一次，並與時間混合交替進行。因此，特徵組合本身還可跨時間混合，之後再次組合。對橫斷面特徵如此密集的資料，受檢驗的正是這種重複操作。

此處設定堆疊兩個區塊，隱藏維度為 32，小於循環模型的狀態，理由相同：值得擬合的容量受合格窗口數量限制。

**學習目標。** 完成本 Notebook 後，你將能夠：

- 描述兩種混合運算，指出各自作用於窗口的哪條軸線，以及經過兩個區塊後，資訊可以如何傳遞。
- 說明 TSMixer 的特徵混合，與另外兩個模型結合特徵的方式有何不同，儘管它們也都會結合特徵。
- 從成本與訓練時必須維持的資訊流，解釋為何沒有遞迴的架構可能更適合長窗口。
- 從宣告的設定讀出 epoch 排程，指出一次執行會為該設定發布多少個可評分模型。
<!-- ML4T_ZH_END c0000-p001 -->

**A neural fit has a meaningful state at every epoch**, in the way a boosted model has one at
every iteration and a linear fit does not. An **epoch** is one pass over the training windows.
Each configuration here trains for 100 of them and saves its weights every 5, so it publishes
twenty scoreable models rather than one, each registered with its own identity. The count that
matters downstream is configurations times checkpoints.

**Book reference**: Chapter 13. Chapter 6, Section 6.7 (Search accounting and run logging)
introduces the run log this notebook writes to.

**Prerequisites**: [`03_financial_features`](03_financial_features.ipynb) and
[`04_model_based_features`](04_model_based_features.ipynb) have written the feature matrices, and
[`05_evaluation`](05_evaluation.ipynb) has established the walk-forward folds.

**What it writes**: one training run per configuration and one complete validation prediction set
per configuration and epoch checkpoint, in `run_log/registry.db` and under `run_log/training/`
and `run_log/predictions/`, grouped under a named population.
[`15_model_analysis`](15_model_analysis.ipynb) compares that population against the other
families and [`16_backtest`](16_backtest.ipynb) backtests every member and selects on validation
backtest Sharpe. **Selection happens there, not here.**

<!-- ML4T_ZH_BEGIN c0000-p002 -->
**神經網路在每個 epoch 都有具意義的狀態**，如同提升式模型每次迭代都有狀態，而線性擬合沒有。**Epoch** 是對訓練窗口完整走過一次。此處每項設定訓練 100 個 epoch，每 5 個儲存一次權重，因此發布二十個可評分模型，而非一個；每個模型都以自身識別資訊登錄。對下游重要的數量是設定數乘以檢查點數。

**書籍參考**：第 13 章。第 6 章，第 6.7 節（搜尋次數核算與執行紀錄）介紹本 Notebook 寫入的執行日誌。

**先備條件**：[`03_financial_features`](03_financial_features.ipynb) 與 [`04_model_based_features`](04_model_based_features.ipynb) 已寫出特徵矩陣，[`05_evaluation`](05_evaluation.ipynb) 已建立滾動向前各折。

**寫出內容**：每項設定各有一次訓練執行紀錄，每項設定與 epoch 檢查點各有一組完整驗證預測；資料寫入 `run_log/registry.db`，以及 `run_log/training/` 和 `run_log/predictions/` 之下，歸屬於具名模型群組。[`15_model_analysis`](15_model_analysis.ipynb) 會將該群組與其他模型類別比較，[`16_backtest`](16_backtest.ipynb) 則回測每位成員，並依驗證回測夏普比率選擇模型。**選擇發生在該處，不在本 Notebook。**
<!-- ML4T_ZH_END c0000-p002 -->


In [ ]:
"""Generate TSMixer validation predictions through the shared research interface."""

import matplotlib.pyplot as plt
import polars as pl
import yaml

from case_studies.research import (
    candidate_set_supersedes,
    open_study,
    plan_models,
    run_model_population,
    supersedes_for_run,
)
from utils.modeling import load_configs
from utils.paths import get_case_study_dir
from utils.style import FIGSIZE, add_message_title, ml4t_palette, show_with_alt, zero_line

In [ ]:
CASE_STUDY_ID = "us_equities_panel"
PRIMARY_LABEL = ""
CONFIG_NAMES = []
COMMON_OVERRIDES = {}
CONFIG_OVERRIDES = {}
POPULATION_NAME = ""
SUPERSEDES_POPULATION = ""
SUPERSEDES_SETS: dict = {}
DEVICE = "cuda"
EXECUTION_TIER = "canonical"
WORKSPACE = ""
PREVIEW_MAX_SYMBOLS = 0
PREVIEW_FOLD_IDS = []
PREVIEW_MAX_TRAIN_SEQUENCES = 0

## 1. Which configurations, and on which label

The menu at `config/training/{label}.yaml` lists the sequence configurations declared for a
label, and this notebook takes the ones whose architecture is `tsmixer`. Each name resolves to a
preset holding the full parameter set - here a 60-session lookback, 100 epochs, a checkpoint
every 5, and dropout on the hidden units. The frame below prints the resolved values.

What each setting a run may pass decides:

- **`CONFIG_NAMES`** empty fits every declared `tsmixer` configuration. A named subset fits only
  those, which is what to do first: at panel scale a full run is hours, and the point of a first
  pass is to find out whether the plumbing works.
- **`COMMON_OVERRIDES`** changes a parameter for every selected configuration, and
  **`CONFIG_OVERRIDES`** changes one named configuration and takes precedence. An override moves
  a training identity, so an overridden run registers beside the published one rather than
  replacing it.
- **`EXECUTION_TIER`** is `canonical` or `preview`. A canonical run fits every eligible window on
  every fold at the published epoch schedule. A preview run has to declare at least one
  reduction and carries it in the identity, so its results can never be compared against
  canonical ones or reach a holdout decision.


<!-- ML4T_ZH_BEGIN c0003-p000 -->
## 1. 哪些設定，以及使用哪個標籤

`config/training/{label}.yaml` 的選單列出為某個標籤宣告的序列模型設定，本 Notebook 選取架構為 `tsmixer` 的項目。各名稱都對應到一組完整參數的預設設定：此處回看 60 個交易日、訓練 100 個 epoch、每 5 個 epoch 設一個檢查點，並對隱藏單元使用 dropout。下方資料框印出解析後數值。

執行時可傳入的各項設定，其用途如下：

- **`CONFIG_NAMES`** 為空時，擬合所有已宣告的 `tsmixer` 設定。若指定名稱子集，就只擬合那些設定。首次執行應先縮小範圍：以此面板規模，完整執行需要數小時；首次嘗試的目的是確認整套流程能運作。
- **`COMMON_OVERRIDES`** 會變更所有選取設定的參數；**`CONFIG_OVERRIDES`** 則變更單一具名設定，且優先適用。覆寫會改變訓練識別資訊，因此覆寫後的執行會在已發布結果旁另行登錄，而非取代它。
- **`EXECUTION_TIER`** 是 `canonical` 或 `preview`。標準執行在每一折，以已發布的 epoch 排程擬合每個合格窗口；預覽執行必須宣告至少一項縮減，並在識別資訊中保留這項縮減，因此結果絕不會與標準結果比較，也不會進入保留樣本決策。
<!-- ML4T_ZH_END c0003-p000 -->

A shortened training schedule is not among the reductions a preview may declare, and
deliberately. This family's preview contract - `SEQUENCE_PREVIEW_FIELDS` in
`case_studies/utils/preview_fields.py` - accepts a narrower universe, a fold subset and a cap on
training sequences, and no epoch count, because a model trained for fewer epochs is a different
model rather than the same one measured sooner. To train a short schedule, pass `n_epochs` in
`COMMON_OVERRIDES`. It moves the training identity, so the result registers beside the published
one, and a run carrying any override publishes neither the canonical population nor the
canonical set names.

<!-- ML4T_ZH_BEGIN c0003-p001 -->
預覽執行可宣告的縮減項目刻意不包含縮短訓練時程。此模型類別的預覽契約 `SEQUENCE_PREVIEW_FIELDS` 位於 `case_studies/utils/preview_fields.py`，允許縮小股票範圍、使用部分折，以及限制訓練序列數量，卻不允許減少 epoch 數，因為訓練較少 epoch 得到的是另一個模型，而非更快量測同一模型。若要使用較短時程訓練，請傳入 `n_epochs`，位置為 `COMMON_OVERRIDES`。這會改變訓練識別資訊，因此結果會登錄在已發布結果旁；帶有任何覆寫設定的執行，既不發布標準模型群組，也不發布標準集合名稱。
<!-- ML4T_ZH_END c0003-p001 -->


In [ ]:
case_dir = get_case_study_dir(CASE_STUDY_ID)
setup = yaml.safe_load((case_dir / "config" / "setup.yaml").read_text())
label = PRIMARY_LABEL or setup["labels"]["primary"]

all_sequence_configs = load_configs(CASE_STUDY_ID, label, family="deep_learning")
published_configs = [
    config
    for config in all_sequence_configs
    if config.get("params", {}).get("architecture") == "tsmixer"
]
published_names = [str(config["config_name"]) for config in published_configs]
selected_names = list(CONFIG_NAMES) if CONFIG_NAMES else published_names
unknown_names = sorted(set(selected_names) - set(published_names))
unknown_overrides = sorted(set(CONFIG_OVERRIDES) - set(selected_names))
if not published_names:
    raise ValueError("The published training menu has no TSMixer configuration")
if unknown_names:
    raise ValueError(f"Unknown TSMixer configurations: {unknown_names}")
if unknown_overrides:
    raise ValueError(f"Overrides supplied for unselected configurations: {unknown_overrides}")
if len(selected_names) != len(set(selected_names)):
    raise ValueError("CONFIG_NAMES contains duplicates")

menu = pl.DataFrame(
    {
        "config_name": [config["config_name"] for config in published_configs],
        "architecture": [config["params"]["architecture"] for config in published_configs],
        "published_params": [str(config.get("params") or {}) for config in published_configs],
        "n_epochs": [config.get("n_epochs") for config in published_configs],
        "checkpoint_interval": [config.get("checkpoint_interval") for config in published_configs],
        "selected": [config["config_name"] in selected_names for config in published_configs],
    }
)
menu

A run that narrows the selection, overrides a parameter or fits on another device produces a
different set of predictions from the one the canonical name stands for. Publishing it under
that name would leave the name meaning two different member sets at two different times, so the
guard below requires such a run to say what to call its own population, and the frozen set names
in Section 6 are withheld from it for the same reason.

<!-- ML4T_ZH_BEGIN c0005-p000 -->
若執行縮小選取範圍、覆寫參數，或改在另一種裝置上擬合，產生的預測集合便與標準名稱所代表者不同。若以同名發布，名稱在不同時間就會代表不同成員集合。因此，下方防護條件要求這種執行自行指定模型群組名稱；第 6 節的固定集合名稱也基於相同理由不提供給它。
<!-- ML4T_ZH_END c0005-p000 -->


In [ ]:
is_published_population = (
    EXECUTION_TIER == "canonical"
    and selected_names == published_names
    and not COMMON_OVERRIDES
    and not CONFIG_OVERRIDES
    and DEVICE == "cuda"
)
if EXECUTION_TIER == "canonical" and not is_published_population and not POPULATION_NAME:
    raise ValueError(
        "this run narrows or overrides what the menu declares, so it cannot publish the canonical "
        "population; pass POPULATION_NAME to give it its own"
    )

Both tiers resolve the study through `open_study`. It reads the labels and features in place
and redirects only writes, so a preview run scores the same inputs a canonical one does and
cannot publish over it. A preview must be given a workspace to write into; a canonical run
leaves `WORKSPACE` empty and regenerates the case study's own artifacts in place.

<!-- ML4T_ZH_BEGIN c0007-p000 -->
兩種執行層級都透過 `open_study` 解析研究。它就地讀取標籤與特徵，只重新導向寫入位置；因此預覽執行使用的輸入與標準執行相同，卻無法覆蓋標準結果。預覽執行必須指定寫入用的工作區；標準執行則讓 `WORKSPACE` 留空，並在案例研究原位置重新產生產出檔案。
<!-- ML4T_ZH_END c0007-p000 -->


In [ ]:
preview_reductions = {}
if PREVIEW_MAX_SYMBOLS:
    preview_reductions["max_symbols"] = int(PREVIEW_MAX_SYMBOLS)
if PREVIEW_FOLD_IDS:
    preview_reductions["folds"] = [int(fold) for fold in PREVIEW_FOLD_IDS]
if PREVIEW_MAX_TRAIN_SEQUENCES:
    preview_reductions["max_train_sequences"] = int(PREVIEW_MAX_TRAIN_SEQUENCES)

study = open_study(CASE_STUDY_ID, execution_tier=EXECUTION_TIER, workspace=WORKSPACE or None)

## 2. Binding the declarations to the data

Each selected TSMixer configuration becomes one request with the declared sequence reductions.

<!-- ML4T_ZH_BEGIN c0009-p000 -->
## 2. 將宣告對應到資料

每項選取的 TSMixer 設定，都會連同宣告的序列縮減條件，成為一項請求。
<!-- ML4T_ZH_END c0009-p000 -->


In [ ]:
requests = []
for config_name in selected_names:
    overrides = {
        "device": DEVICE,
        **COMMON_OVERRIDES,
        **dict(CONFIG_OVERRIDES.get(config_name, {})),
    }
    requests.append(
        study.model(
            family="deep_learning",
            label=label,
            config_name=config_name,
            overrides=overrides,
            execution_tier=EXECUTION_TIER,
            preview_reductions=preview_reductions,
        )
    )
requests = tuple(requests)

request_table = pl.DataFrame(
    {
        "family": [request.family for request in requests],
        "label": [request.label for request in requests],
        "config_name": [request.config_name for request in requests],
        "overrides": [str(request.overrides) for request in requests],
        "execution_tier": [request.execution_tier.value for request in requests],
        "preview_reductions": [str(request.preview_reductions) for request in requests],
    }
)
request_table

## 3. Planning, then fitting

The planner resolves every training and epoch-checkpoint identity before fitting and writes the
canonical checkpoint population first. Execution builds only sequences that follow the declared
observation calendar and excludes
windows that cross missing expected periods. Each epoch checkpoint stores the fitted
preprocessing state, model weights, predictions, and exact eligible-key evidence. A retry reuses
valid candidate-fold checkpoints and recomputes incomplete work.

<!-- ML4T_ZH_BEGIN c0011-p000 -->
## 3. 先規劃，再擬合

規劃器在擬合前解析每項訓練及 epoch 檢查點的識別資訊，並先寫出標準檢查點模型群組。執行時只建立符合宣告觀測日曆的序列，排除跨越預期期間缺口的窗口。每個 epoch 檢查點都儲存擬合後的前處理狀態、模型權重、預測值，以及精確的合格鍵證據。重試時會沿用有效的候選模型與折檢查點，只重新計算未完成的工作。
<!-- ML4T_ZH_END c0011-p000 -->


In [ ]:
plan = plan_models(study, requests=requests)

planned_population = pl.DataFrame(
    {
        "family": [member.family for member in plan.members],
        "config_name": [member.config_name for member in plan.members],
        "checkpoint_kind": [member.checkpoint_kind for member in plan.members],
        "checkpoint_value": [member.checkpoint_value for member in plan.members],
        "training_hash": [member.training_hash for member in plan.members],
        "prediction_hash": [member.prediction_hash for member in plan.members],
    }
)
planned_population

`run_model_population` takes the plan, writes the population down, fits every member and then
checks that what came out is what was declared. The same call serves both tiers: a canonical run
registers an immutable population that the later notebooks bind to, and a preview run gets a
declaration that is verified and then discarded with its workspace, so no notebook here has to
branch on the tier to decide what to publish.

`SUPERSEDES_POPULATION` names the population hash this run replaces. A population is a set of
prediction identities, so anything that moves a training identity - a changed preset as much as a
changed menu - produces a different population under the same name, and the registry refuses to
write it without being told which snapshot it supersedes. Leaving it empty is right for a first
run and for a reader's clean clone, and `supersedes_for_run` withholds a declared hash wherever
offering it would be refused.

<!-- ML4T_ZH_BEGIN c0013-p000 -->
`run_model_population` 取得計畫、記錄模型群組、擬合每位成員，最後檢查實際產出是否符合宣告。同一呼叫服務兩種執行層級：標準執行登錄不可變的模型群組，供後續 Notebook 綁定；預覽執行則取得經驗證的宣告，之後隨其工作區一起捨棄。因此，此處 Notebook 不必依執行層級分支，才知道該發布什麼。

`SUPERSEDES_POPULATION` 指定本次執行要取代的群組雜湊值。模型群組是一組預測識別資訊；凡使訓練識別資訊改變的事項，不論是預設設定或選單變更，都會在同一名稱下產生不同群組。登錄系統若未獲告知要取代哪個快照，就會拒絕寫入。首次執行或讀者剛取得的乾淨副本，應讓此值留空；`supersedes_for_run` 會在提供已宣告雜湊值將遭拒絕的情況下省略它。
<!-- ML4T_ZH_END c0013-p000 -->


In [ ]:
population_name = POPULATION_NAME or "us-equities-tsmixer-checkpoints-v1"
execution, official_population = run_model_population(
    study,
    plan,
    population_name=population_name,
    supersedes=supersedes_for_run(
        study,
        population_name=population_name,
        declared=SUPERSEDES_POPULATION,
        execution_tier=EXECUTION_TIER,
    ),
)

print(f"population {official_population.name}: {len(official_population.members)} prediction sets")

## 4. What was actually fitted

These rows expose the feature, fold, sequence, runtime, model, and checkpoint settings used by
the runner, including defaults that were not repeated in the notebook parameters.

<!-- ML4T_ZH_BEGIN c0015-p000 -->
## 4. 實際擬合的內容

這些資料列呈現執行器使用的特徵、折、序列、執行環境、模型與檢查點設定，也包含 Notebook 參數未重述的預設值。
<!-- ML4T_ZH_END c0015-p000 -->


In [ ]:
resolved_rows = []
for run in execution.runs:
    spec = run.training.spec()
    computation = spec["computation"]
    model = computation["model"]
    resolved_rows.append(
        {
            "config_name": spec["config_name"],
            "architecture": model["params"]["architecture"],
            "features": len(computation["feature_names"]),
            "folds": computation["expected_prediction_keys"]["n_folds"],
            "eligible_rows": computation["expected_prediction_keys"]["n_rows"],
            "lookback": model["params"]["lookback"],
            "device": computation["numerics"]["device"],
            "n_epochs": model["params"]["n_epochs"],
            "checkpoints": [item["value"] for item in computation["checkpoint_schedule"]],
            "training_hash": run.training.hash,
        }
    )

resolved_table = pl.DataFrame(resolved_rows).sort("config_name")
resolved_table

## 5. What came out

Each catalog row is one complete validation prediction set for one training identity and epoch.
Downstream notebooks filter these rows with Polars and pass the selected table directly to
backtesting. The hashes remain visible for exact provenance and artifact reads.

<!-- ML4T_ZH_BEGIN c0017-p000 -->
## 5. 產生的結果

目錄中的每列，代表某個訓練識別資訊與 epoch 對應的一組完整驗證預測。下游 Notebook 以 Polars 篩選這些列，再將選取的表格直接交給回測。雜湊值持續顯示，便於精確追溯來源與讀取產出檔案。
<!-- ML4T_ZH_END c0017-p000 -->


In [ ]:
catalog_columns = [
    "family",
    "config_name",
    "label",
    "split",
    "checkpoint_kind",
    "checkpoint_value",
    "execution_tier",
    "complete",
    "ic_mean",
    "training_hash",
    "prediction_hash",
]
catalog_rows = execution.catalog_rows.select(
    column for column in catalog_columns if column in execution.catalog_rows.columns
).sort("config_name", "checkpoint_value", "prediction_hash")
catalog_rows

A prediction set can be registered complete and still have scored no dates. Cross-sectional
information coefficient needs a minimum number of names quoted on a date before the ranking on
that date means anything, so a universe whose stocks do not overlap in time yields no scorable
dates and a null IC at every checkpoint while every coverage check passes. That is a run which
reports nothing and looks successful, so it is asserted on rather than left to be noticed.

<!-- ML4T_ZH_BEGIN c0019-p000 -->
預測集合即使登錄為完整，也可能沒有任何可評分日期。橫斷面資訊係數要求某日期至少有足夠股票報價，該日的排序才有意義；若股票交易期間彼此不重疊，就沒有可評分日期，所有檢查點的資訊係數均為空值，卻仍會通過涵蓋率檢查。這種執行看似成功，實際上沒有報告任何結果，因此要以斷言檢查，而非等人注意到。
<!-- ML4T_ZH_END c0019-p000 -->


In [ ]:
scored = execution.catalog_rows.select("config_name", "checkpoint_value", "ic_mean", "ic_n_days")
unscored = scored.filter(pl.col("ic_n_days").is_null() | (pl.col("ic_n_days") <= 0))
if not unscored.is_empty():
    raise RuntimeError(f"prediction sets scored no dates: {unscored.to_dicts()}")
scored

### Where more training stopped helping

Each line traces one configuration's validation information coefficient as epochs are added to
it. This is the figure the checkpoint dimension exists to produce, and it separates two things a
single end-of-training number cannot.

A line that rises and then falls has an interior optimum: the model was still learning, then
began fitting the training windows at the expense of the validation folds. That is the evidence about whether alternating
time and feature mixing over two blocks has more capacity than the number of eligible
windows supports.
A line that wanders around zero without trend never had anything to learn, and its highest point
is wherever the noise happened to peak. Both produce a respectable-looking maximum, which is why
the curve rather than the maximum is what to read.

Nothing here selects a checkpoint. Every one of them is registered as its own candidate, and
which one a strategy would use is decided by validation backtest Sharpe in
[`16_backtest`](16_backtest.ipynb).

<!-- ML4T_ZH_BEGIN c0021-p000 -->
### 增加訓練不再有幫助的位置

每條線追蹤某項設定隨 epoch 增加的驗證資訊係數。檢查點維度正是為了產生這張圖；它能區分只看訓練結束數值無法分辨的兩種情況。

若曲線先升後降，就有內部最佳點：模型起初仍在學習，之後開始以犧牲驗證折表現為代價，擬合訓練窗口。這是判斷在兩個區塊間交替混合時間與特徵，是否超出合格窗口數量所能支撐模型容量的證據。若曲線在零附近無方向地游移，模型一開始就沒有可學的內容，其最高點只是雜訊碰巧達到峰值的位置。兩者都可能產生看似不錯的最大值，因此應解讀曲線，而非只看最大值。

此處不選擇檢查點。每個檢查點都單獨登錄為候選項；策略要使用哪一個，則由 [`16_backtest`](16_backtest.ipynb) 的驗證回測夏普比率決定。
<!-- ML4T_ZH_END c0021-p000 -->


In [ ]:
curves = scored.sort("config_name", "checkpoint_value")
config_names = curves.get_column("config_name").unique(maintain_order=True).to_list()
# `ml4t_palette` returns a list of that many colours, so it is called once and indexed.
palette = ml4t_palette(len(config_names), categorical=True)

fig, ax = plt.subplots(figsize=FIGSIZE["single"])
for index, config_name in enumerate(config_names):
    series = curves.filter(pl.col("config_name") == config_name)
    ax.plot(
        series.get_column("checkpoint_value"),
        series.get_column("ic_mean"),
        marker="o",
        markersize=4,
        lw=1.4,
        color=palette[index],
        label=config_name,
    )
zero_line(ax)
ax.set_xlabel("Training epochs")
ax.set_ylabel("Mean validation IC")
ax.legend(fontsize=8, frameon=False)
add_message_title(
    ax,
    "Mean validation IC against training epoch",
    subtitle="One line per configuration, over the epochs the schedule checkpoints at",
)
# The alt text counts rather than asserts: whether a curve turns over is the question the figure
# exists to answer, and a line described as peaking when it does not is a claim the data refutes.
_peaks = (
    curves.group_by("config_name")
    .agg(
        peak=pl.col("checkpoint_value").sort_by("ic_mean", descending=True).first(),
        first=pl.col("checkpoint_value").min(),
        last=pl.col("checkpoint_value").max(),
    )
    .with_columns(
        interior=pl.col("peak").is_between(pl.col("first"), pl.col("last"), closed="none")
    )
)
_n_interior = int(_peaks.get_column("interior").sum())
show_with_alt(
    fig,
    "A line chart of mean validation information coefficient against training epoch, one line per "
    "configuration, with a dashed line at zero. Counted from the underlying frame, "
    f"{_n_interior} of {_peaks.height} configurations reach their highest information coefficient "
    "at an epoch that is neither the first nor the last, which is what an interior optimum looks "
    "like on this chart.",
)

In [ ]:
coverage_rows = []
for run in execution.runs:
    if not run.training.complete:
        raise RuntimeError(f"Incomplete training result: {run.training.hash}")
    for prediction in run.predictions:
        record = prediction.registry_record()
        coverage = prediction.coverage()
        if not prediction.complete or coverage is None or coverage["status"] != "complete":
            raise RuntimeError(f"Incomplete prediction result: {prediction.hash}")
        coverage_rows.append(
            {
                "config_name": run.training.spec()["config_name"],
                "checkpoint": record["checkpoint_value"],
                "training_hash": run.training.hash,
                "prediction_hash": prediction.hash,
                "coverage_status": coverage["status"],
                "expected_rows": coverage["n_expected"],
                "actual_rows": coverage["n_actual"],
                "training_artifacts": len(run.training.artifacts()),
                "prediction_artifacts": len(prediction.artifacts()),
            }
        )

coverage_table = pl.DataFrame(coverage_rows).sort("config_name", "checkpoint")
coverage_table

In [ ]:
execution_diagnostics = pl.DataFrame(execution.diagnostics)
execution_diagnostics

## 6. Naming the sets the later notebooks open

A canonical default CUDA run freezes what it produced under two stable names, and preview or
customized canonical requests publish neither.

The first name is the **full set**: every prediction row this run returned, which
[`16_backtest`](16_backtest.ipynb) backtests member by member.

The second is the **bounded diagnostic set**, and it is bounded hard.
[`15_model_analysis`](15_model_analysis.ipynb) loads every diagnostic member's raw prediction
frame and holds them all while it joins them pairwise; one frame on this panel is over seven
million rows and about 225 MB in memory, so a set that grew with the checkpoint count would not fit
beside the other seven families'. The bound is the last checkpoint of each published
configuration - one member here, because the menu declares one TSMixer configuration. The
epoch dimension is still read, in the learning-curve figure above, which is drawn from registry
metrics rather than from raw frames.

<!-- ML4T_ZH_BEGIN c0025-p000 -->
## 6. 為後續 Notebook 開啟的集合命名

預設 CUDA 裝置上的標準執行，會將產出固定於兩個穩定名稱之下；預覽或自訂的標準執行，都不發布這些名稱。

第一個名稱是**完整集合**：本次執行回傳的每列預測，由 [`16_backtest`](16_backtest.ipynb) 逐一回測成員。

第二個名稱是**受限診斷集合**，有嚴格上限。[`15_model_analysis`](15_model_analysis.ipynb) 會載入每位診斷成員的原始預測資料框，並在逐對連結期間全部保留；此面板的一份資料框超過七百萬列，記憶體約占 225 MB。若集合成員數隨檢查點增加，就無法與其他七種模型類別的資料同時放入記憶體。上限為各項已發布設定的最後檢查點；此處只有一位成員，因為選單只宣告一項 TSMixer 設定。epoch 維度仍可從上方學習曲線讀取；該圖來自登錄資料的指標，而非原始資料框。
<!-- ML4T_ZH_END c0025-p000 -->


In [ ]:
set_rows = []
if is_published_population:
    label_name = label.replace("_", "-")
    full_set_name = f"us-equities-{label_name}-tsmixer-v1"
    full_set = study.predictions.freeze(
        execution.catalog_rows,
        name=full_set_name,
        supersedes=candidate_set_supersedes(
            study, name=full_set_name, declared=SUPERSEDES_SETS.get(full_set_name, "")
        ),
    )
    diagnostic_rows = execution.catalog_rows.filter(
        # `.fill_null(True)` covers a family that publishes no checkpoint value at all, where the
        # comparison is null rather than false and would otherwise empty the frame.
        (
            pl.col("checkpoint_value") == pl.col("checkpoint_value").max().over("config_name")
        ).fill_null(True)
    )
    diagnostic_set_name = f"us-equities-{label_name}-tsmixer-diagnostics-v1"
    diagnostic_set = study.predictions.freeze(
        diagnostic_rows,
        name=diagnostic_set_name,
        supersedes=candidate_set_supersedes(
            study, name=diagnostic_set_name, declared=SUPERSEDES_SETS.get(diagnostic_set_name, "")
        ),
    )
    set_rows = [
        {
            "role": "backtest population",
            "set_name": full_set.name,
            "members": len(full_set.members),
        },
        {
            "role": "bounded diagnostics",
            "set_name": diagnostic_set.name,
            "members": len(diagnostic_set.members),
        },
    ]
compatible_sets = pl.DataFrame(
    set_rows,
    schema={"role": pl.String, "set_name": pl.String, "members": pl.Int64},
)
compatible_sets

`15_model_analysis` reopens both names: the full set to confirm the run filled every member it
promised, and the diagnostic set to read raw predictions. `16_backtest` passes every full-set
catalog row to the shared backtest runner. Neither the metrics here nor the ones there choose a
configuration or a checkpoint; selection is on validation backtest Sharpe in `16_backtest`.

<!-- ML4T_ZH_BEGIN c0027-p000 -->
`15_model_analysis` 會重新開啟兩個名稱：完整集合用於確認本次執行完成承諾的每位成員，診斷集合用於讀取原始預測。`16_backtest` 將完整集合目錄中的每列傳給共用回測執行器。此處或該處的指標都不決定設定或檢查點；模型選擇在 `16_backtest` 依驗證回測夏普比率進行。
<!-- ML4T_ZH_END c0027-p000 -->


## What to notice

**Three architectures, one window, one set of folds.** NLinear applies a single linear map, the
LSTM walks the window carrying state, and this mixes along both axes. Any difference between
them is the architecture, because nothing else varies - which is what makes the comparison worth
having, and is also why none of the three chooses anything here.

**What is being tested is repeated, explicit feature mixing, not feature mixing as such.** Both
of the other models combine features once. If alternating that with temporal mixing block after
block is worth anything on this panel, this is where it shows, and the place to look is against
the LSTM rather than against NLinear.

**A checkpoint is part of a configuration.** Twenty per configuration, each registered
separately, for the reason `09_dl_nlinear` gives.

**Known limitations.** Windows are built only where sessions are unbroken. Everything measured
here is ranking accuracy on validation folds read many times over, and the comparison between
the three families is settled on validation backtest Sharpe in
[`16_backtest`](16_backtest.ipynb), not here.

**Next**: [`12_dl_weekly`](12_dl_weekly.ipynb) keeps these models and changes the sampling grid.

<!-- ML4T_ZH_BEGIN c0028-p000 -->
## 觀察重點

**三種架構、相同窗口、相同折集合。** NLinear 套用單一線性映射，LSTM 帶著狀態走過窗口，TSMixer 則沿兩個軸向混合。其間的差異來自架構，因為其他條件都相同；這使比較具有價值，也正是三者都不在此處做選擇的原因。

**檢驗的是重複、明確的特徵混合，而非特徵混合本身。** 另外兩種模型都只結合一次特徵。若逐區塊交替進行特徵與時間混合，對此面板確有價值，結果會在此顯現；比較時應看 LSTM，而非 NLinear。

**檢查點是設定的一部分。** 每項設定有二十個檢查點，各自登錄，理由見 `09_dl_nlinear`。

**已知限制。** 只在交易時段連續時建立窗口。此處量測的是經過多次檢視之驗證折的排序準確度；三種模型類別的比較，會在 [`16_backtest`](16_backtest.ipynb) 依驗證回測夏普比率定案，而非在此處。

**下一步**：[`12_dl_weekly`](12_dl_weekly.ipynb) 保留這些模型，改變取樣網格。
<!-- ML4T_ZH_END c0028-p000 -->
